# Public Data, Capacity, and Cloud Integration

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lolusername/CST4714_OER/blob/main/course_materials/notebooks/06_public_data_capacity_integration.ipynb)

[View on GitHub](https://github.com/lolusername/CST4714_OER/blob/main/course_materials/notebooks/06_public_data_capacity_integration.ipynb)

Click **Open in Colab** to open this notebook directly. Run the cells in order.

This notebook uses a small teaching snapshot of the U.S. Cybersecurity and
Infrastructure Security Agency's Known Exploited Vulnerabilities catalog. It
connects four skills: evaluating a source, checking data quality, reasoning about
distribution, and loading records without creating duplicates.

**By the end, you will be able to:**

- identify source, retrieval, transformation, and use limits before importing;
- check nulls, duplicates, identifiers, and dates;
- compare cardinality, frequency, monotonicity, and query targeting;
- explain range and hashed distribution with measured results;
- load records idempotently into SQLite, Atlas, or PostgreSQL; and
- verify more than a successful connection or insert count.

The SQLite path uses only Python's standard library and the embedded data. After
opening the notebook, it needs no package download or database account. Colab
itself is an online service. A local Jupyter session can run the SQLite path
offline. Optional cloud paths need a network connection and install their own
driver only when selected.

## Resource and License Boundary

The notebook prose and code are course OER. The CISA records come from an official
U.S. government feed and are not represented as original course data. The
embedded snapshot preserves source metadata and a description of the field
selection. It is a compact classroom fixture, not a current vulnerability-
management source.

Official feed:
<https://www.cisa.gov/sites/default/files/feeds/known_exploited_vulnerabilities.json>

In [ ]:
from bisect import bisect_right
from collections import Counter
from datetime import date
from getpass import getpass
from hashlib import sha256
from ipaddress import IPv4Address
import json
import re
import sqlite3
import urllib.request
from uuid import uuid4

print("Python's built-in libraries are ready. No cloud connection was opened.")

## 1. Inspect the Source and Its Records

`USE_LIVE_FEED` is `False` by default. That makes the class result reproducible
and keeps the notebook usable during an outage. Change it to `True` only when you
intend to inspect the current official feed. Current results will differ from the
versioned teaching snapshot.

In [ ]:
OFFLINE_SNAPSHOT = json.loads(r'''{"title": "CISA Catalog of Known Exploited Vulnerabilities", "catalogVersion": "2026.07.10", "sourceDateReleased": "2026-07-10T17:00:25.7327Z", "retrievedAt": "2026-07-13T06:19:55.069283+00:00", "sourceUrl": "https://www.cisa.gov/sites/default/files/feeds/known_exploited_vulnerabilities.json", "selection": "first 75 source records; documented fields only", "count": 75, "vulnerabilities": [{"cveID": "CVE-2026-56291", "vendorProject": "Balbooa", "product": "Forms", "vulnerabilityName": "Balbooa Forms Unrestricted Upload of File with Dangerous Type Vulnerability", "dateAdded": "2026-07-10", "dueDate": "2026-07-13", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-434"]}, {"cveID": "CVE-2026-48939", "vendorProject": "iCagenda", "product": "iCagenda", "vulnerabilityName": "iCagenda Unrestricted Upload of File with Dangerous Type Vulnerability", "dateAdded": "2026-07-10", "dueDate": "2026-07-13", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-434"]}, {"cveID": "CVE-2026-48908", "vendorProject": "JoomShaper", "product": "SP Page Builder", "vulnerabilityName": "JoomShaper SP Page Builder Unrestricted Upload of File with Dangerous Type Vulnerability", "dateAdded": "2026-07-07", "dueDate": "2026-07-10", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-434"]}, {"cveID": "CVE-2026-55255", "vendorProject": "Langflow", "product": "Langflow", "vulnerabilityName": "Langflow Authorization Bypass Through User-Controlled Key Vulnerability", "dateAdded": "2026-07-07", "dueDate": "2026-07-10", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-639"]}, {"cveID": "CVE-2026-56290", "vendorProject": "Joomlack", "product": "Page Builder", "vulnerabilityName": "Joomlack Page Builder Improper Access Control Vulnerability", "dateAdded": "2026-07-07", "dueDate": "2026-07-10", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-284"]}, {"cveID": "CVE-2026-48282", "vendorProject": "Adobe", "product": "ColdFusion", "vulnerabilityName": "Adobe ColdFusion Path Traversal Vulnerability", "dateAdded": "2026-07-07", "dueDate": "2026-07-10", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-22"]}, {"cveID": "CVE-2026-45659", "vendorProject": "Microsoft", "product": "SharePoint Server", "vulnerabilityName": "Microsoft SharePoint Server Deserialization of Untrusted Data Vulnerability", "dateAdded": "2026-07-01", "dueDate": "2026-07-04", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-502"]}, {"cveID": "CVE-2026-48558", "vendorProject": "SimpleHelp ", "product": "SimpleHelp", "vulnerabilityName": "SimpleHelp Authentication Bypass Vulnerability", "dateAdded": "2026-06-29", "dueDate": "2026-07-02", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-347"]}, {"cveID": "CVE-2026-12569", "vendorProject": "PTC", "product": "Windchill and FlexPLM", "vulnerabilityName": "PTC Windchill and FlexPLM Improper Input Validation Vulnerability", "dateAdded": "2026-06-25", "dueDate": "2026-06-28", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-20", "CWE-502"]}, {"cveID": "CVE-2026-20230", "vendorProject": "Cisco", "product": "Unified Communications Manager", "vulnerabilityName": "Cisco Unified Communications Manager Server-Side Request Forgery (SSRF) Vulnerability", "dateAdded": "2026-06-25", "dueDate": "2026-06-28", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-918"]}, {"cveID": "CVE-2025-67038", "vendorProject": "Lantronix", "product": "EDS5000", "vulnerabilityName": "Lantronix EDS5000 Code Injection Vulnerability", "dateAdded": "2026-06-23", "dueDate": "2026-06-26", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-78", "CWE-94"]}, {"cveID": "CVE-2026-34910", "vendorProject": "Ubiquiti", "product": "UniFi OS", "vulnerabilityName": "Ubiquiti UniFi OS Improper Input Validation Vulnerability", "dateAdded": "2026-06-23", "dueDate": "2026-06-26", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-20"]}, {"cveID": "CVE-2026-34909", "vendorProject": "Ubiquiti", "product": "UniFi OS", "vulnerabilityName": "Ubiquiti UniFi OS Path Traversal Vulnerability", "dateAdded": "2026-06-23", "dueDate": "2026-06-26", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-22"]}, {"cveID": "CVE-2026-34908", "vendorProject": "Ubiquiti", "product": "UniFi OS", "vulnerabilityName": "Ubiquiti UniFi OS Improper Access Control Vulnerability", "dateAdded": "2026-06-23", "dueDate": "2026-06-26", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-284"]}, {"cveID": "CVE-2026-20253", "vendorProject": "Splunk", "product": "Enterprise", "vulnerabilityName": "Splunk Enterprise Missing Authentication for Critical Function Vulnerability", "dateAdded": "2026-06-18", "dueDate": "2026-06-21", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-306"]}, {"cveID": "CVE-2026-48907", "vendorProject": "Widget Factory", "product": "Joomla Content Editor ", "vulnerabilityName": "Widget Factory Joomla Content Editor Improper Access Control Vulnerability", "dateAdded": "2026-06-16", "dueDate": "2026-06-19", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-284"]}, {"cveID": "CVE-2026-54420", "vendorProject": "LiteSpeed", "product": "cPanel Plugin", "vulnerabilityName": "LiteSpeed cPanel Plugin UNIX Symbolic Link (Symlink) Following Vulnerability", "dateAdded": "2026-06-15", "dueDate": "2026-06-18", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-61"]}, {"cveID": "CVE-2026-20262", "vendorProject": "Cisco", "product": "Catalyst SD-WAN Manager", "vulnerabilityName": "Cisco Catalyst SD-WAN Manager Directory or Path Traversal Vulnerability", "dateAdded": "2026-06-15", "dueDate": "2026-06-29", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-22"]}, {"cveID": "CVE-2026-35273", "vendorProject": "Oracle", "product": " PeopleSoft Enterprise PeopleTools", "vulnerabilityName": "Oracle PeopleSoft Enterprise PeopleTools Missing Authentication for Critical Function Vulnerability", "dateAdded": "2026-06-12", "dueDate": "2026-06-15", "knownRansomwareCampaignUse": "Known", "cwes": ["CWE-306"]}, {"cveID": "CVE-2026-10520", "vendorProject": "Ivanti", "product": "Sentry", "vulnerabilityName": "Ivanti Sentry OS Command Injection Vulnerability", "dateAdded": "2026-06-11", "dueDate": "2026-06-14", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-78"]}, {"cveID": "CVE-2026-11645", "vendorProject": "Google", "product": "Chromium V8", "vulnerabilityName": "Google Chromium V8 Out-of-Bounds Read and Write Vulnerability", "dateAdded": "2026-06-09", "dueDate": "2026-06-23", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-787", "CWE-125"]}, {"cveID": "CVE-2026-7473", "vendorProject": "Arista", "product": "Extensible Operating System", "vulnerabilityName": "Arista Extensible Operating System Incomplete Comparison with Missing Factors Vulnerability", "dateAdded": "2026-06-09", "dueDate": "2026-06-23", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-1023"]}, {"cveID": "CVE-2026-20245", "vendorProject": "Cisco", "product": "Catalyst SD-WAN Manager", "vulnerabilityName": "Cisco Catalyst SD-WAN Manager Improper Encoding or Escaping of Output Vulnerability", "dateAdded": "2026-06-09", "dueDate": "2026-06-23", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-116"]}, {"cveID": "CVE-2026-42271", "vendorProject": "BerriAI", "product": "LiteLLM", "vulnerabilityName": "BerriAI LiteLLM Command Injection Vulnerability", "dateAdded": "2026-06-08", "dueDate": "2026-06-22", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-78", "CWE-77"]}, {"cveID": "CVE-2026-50751", "vendorProject": "Check Point", "product": "Security Gateway", "vulnerabilityName": "Check Point Security Gateway Improper Authentication Vulnerability", "dateAdded": "2026-06-08", "dueDate": "2026-06-11", "knownRansomwareCampaignUse": "Known", "cwes": ["CWE-287"]}, {"cveID": "CVE-2026-28318", "vendorProject": "SolarWinds", "product": "Serv-U", "vulnerabilityName": "SolarWinds Serv-U Uncontrolled Resource Consumption Vulnerability", "dateAdded": "2026-06-05", "dueDate": "2026-06-19", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-400"]}, {"cveID": "CVE-2026-45247", "vendorProject": "Mirasvit", "product": "Mirasvit Full Page Cache Warmer", "vulnerabilityName": "Mirasvit Full Page Cache Warmer Deserialization of Untrusted Data Vulnerability", "dateAdded": "2026-06-03", "dueDate": "2026-06-06", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-502"]}, {"cveID": "CVE-2022-0492", "vendorProject": "Linux", "product": "Kernel", "vulnerabilityName": "Linux Kernel Improper Authentication Vulnerability", "dateAdded": "2026-06-02", "dueDate": "2026-06-05", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-287", "CWE-862"]}, {"cveID": "CVE-2025-48595", "vendorProject": "Android", "product": "Framework", "vulnerabilityName": "Android Framework Integer Overflow Vulnerability", "dateAdded": "2026-06-02", "dueDate": "2026-06-05", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-190"]}, {"cveID": "CVE-2024-21182", "vendorProject": "Oracle", "product": "WebLogic Server", "vulnerabilityName": "Oracle WebLogic Server Unspecified Vulnerability", "dateAdded": "2026-06-01", "dueDate": "2026-06-04", "knownRansomwareCampaignUse": "Unknown", "cwes": []}, {"cveID": "CVE-2026-0257", "vendorProject": "Palo Alto Networks", "product": "PAN-OS", "vulnerabilityName": "Palo Alto Networks PAN-OS Authentication Bypass Vulnerability", "dateAdded": "2026-05-29", "dueDate": "2026-06-01", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-565"]}, {"cveID": "CVE-2026-48027", "vendorProject": "Nx", "product": "Nx Console", "vulnerabilityName": "Nx Console Embedded Malicious Code Vulnerability", "dateAdded": "2026-05-27", "dueDate": "2026-06-10", "knownRansomwareCampaignUse": "Known", "cwes": ["CWE-506"]}, {"cveID": "CVE-2026-45321", "vendorProject": "TanStack", "product": "TanStack", "vulnerabilityName": "TanStack Unspecified Vulnerability", "dateAdded": "2026-05-27", "dueDate": "2026-06-10", "knownRansomwareCampaignUse": "Known", "cwes": []}, {"cveID": "CVE-2026-8398", "vendorProject": "Daemon", "product": "Daemon Tools Lite", "vulnerabilityName": "Daemon Tools Lite Embedded Malicious Code Vulnerability", "dateAdded": "2026-05-27", "dueDate": "2026-05-30", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-506"]}, {"cveID": "CVE-2026-48172", "vendorProject": "LiteSpeed", "product": "cPanel Plugin", "vulnerabilityName": "LiteSpeed cPanel Plugin Privilege Escalation Vulnerability", "dateAdded": "2026-05-26", "dueDate": "2026-05-29", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-266"]}, {"cveID": "CVE-2026-9082", "vendorProject": "Drupal", "product": "Core", "vulnerabilityName": "Drupal Core SQL Injection Vulnerability", "dateAdded": "2026-05-22", "dueDate": "2026-05-27", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-89"]}, {"cveID": "CVE-2025-34291", "vendorProject": "Langflow", "product": "Langflow", "vulnerabilityName": "Langflow Origin Validation Error Vulnerability", "dateAdded": "2026-05-21", "dueDate": "2026-06-04", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-346"]}, {"cveID": "CVE-2026-34926", "vendorProject": "Trend Micro", "product": "Apex One", "vulnerabilityName": "Trend Micro Apex One (On-Premise) Directory Traversal Vulnerability", "dateAdded": "2026-05-21", "dueDate": "2026-06-04", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-23"]}, {"cveID": "CVE-2008-4250", "vendorProject": "Microsoft", "product": "Windows", "vulnerabilityName": "Microsoft Windows Buffer Overflow Vulnerability", "dateAdded": "2026-05-20", "dueDate": "2026-06-03", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-94"]}, {"cveID": "CVE-2009-1537", "vendorProject": "Microsoft", "product": "DirectX", "vulnerabilityName": "Microsoft DirectX NULL Byte Overwrite Vulnerability", "dateAdded": "2026-05-20", "dueDate": "2026-06-03", "knownRansomwareCampaignUse": "Unknown", "cwes": []}, {"cveID": "CVE-2009-3459", "vendorProject": "Adobe", "product": "Acrobat and Reader", "vulnerabilityName": "Adobe Acrobat and Reader Heap-Based Buffer Overflow Vulnerability", "dateAdded": "2026-05-20", "dueDate": "2026-06-03", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-119"]}, {"cveID": "CVE-2010-0249", "vendorProject": "Microsoft", "product": "Internet Explorer", "vulnerabilityName": "Microsoft Internet Explorer Use-After-Free Vulnerability", "dateAdded": "2026-05-20", "dueDate": "2026-06-03", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-416"]}, {"cveID": "CVE-2010-0806", "vendorProject": "Microsoft", "product": "Internet Explorer", "vulnerabilityName": "Microsoft Internet Explorer Use-After-Free Vulnerability", "dateAdded": "2026-05-20", "dueDate": "2026-06-03", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-399"]}, {"cveID": "CVE-2026-41091", "vendorProject": "Microsoft", "product": "Defender", "vulnerabilityName": "Microsoft Defender Link Following Vulnerability", "dateAdded": "2026-05-20", "dueDate": "2026-06-03", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-59"]}, {"cveID": "CVE-2026-45498", "vendorProject": "Microsoft", "product": "Defender", "vulnerabilityName": "Microsoft Defender Denial of Service Vulnerability", "dateAdded": "2026-05-20", "dueDate": "2026-06-03", "knownRansomwareCampaignUse": "Unknown", "cwes": []}, {"cveID": "CVE-2026-42897", "vendorProject": "Microsoft", "product": "Microsoft", "vulnerabilityName": "Microsoft Exchange Server Cross-Site Scripting Vulnerability", "dateAdded": "2026-05-15", "dueDate": "2026-05-29", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-79"]}, {"cveID": "CVE-2026-20182", "vendorProject": "Cisco", "product": "Catalyst SD-WAN", "vulnerabilityName": "Cisco Catalyst SD-WAN Controller Authentication Bypass Vulnerability", "dateAdded": "2026-05-14", "dueDate": "2026-05-17", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-287"]}, {"cveID": "CVE-2026-42208", "vendorProject": "BerriAI", "product": "LiteLLM", "vulnerabilityName": "BerriAI LiteLLM SQL Injection Vulnerability", "dateAdded": "2026-05-08", "dueDate": "2026-05-11", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-89"]}, {"cveID": "CVE-2026-6973", "vendorProject": "Ivanti", "product": "Endpoint Manager Mobile (EPMM)", "vulnerabilityName": "Ivanti Endpoint Manager Mobile (EPMM) Improper Input Validation Vulnerability", "dateAdded": "2026-05-07", "dueDate": "2026-05-10", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-20"]}, {"cveID": "CVE-2026-0300", "vendorProject": "Palo Alto Networks", "product": "PAN-OS", "vulnerabilityName": "Palo Alto Networks PAN-OS Out-of-bounds Write Vulnerability", "dateAdded": "2026-05-06", "dueDate": "2026-05-09", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-787"]}, {"cveID": "CVE-2026-31431", "vendorProject": "Linux", "product": "Kernel", "vulnerabilityName": "Linux Kernel Incorrect Resource Transfer Between Spheres Vulnerability", "dateAdded": "2026-05-01", "dueDate": "2026-05-15", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-669"]}, {"cveID": "CVE-2026-41940", "vendorProject": "WebPros", "product": "cPanel & WHM and WP2 (WordPress Squared)", "vulnerabilityName": "WebPros cPanel & WHM and WP2 (WordPress Squared) Missing Authentication for Critical Function Vulnerability", "dateAdded": "2026-04-30", "dueDate": "2026-05-03", "knownRansomwareCampaignUse": "Known", "cwes": ["CWE-306"]}, {"cveID": "CVE-2024-1708", "vendorProject": "ConnectWise", "product": "ScreenConnect", "vulnerabilityName": "ConnectWise ScreenConnect Path Traversal Vulnerability", "dateAdded": "2026-04-28", "dueDate": "2026-05-12", "knownRansomwareCampaignUse": "Known", "cwes": ["CWE-22"]}, {"cveID": "CVE-2026-32202", "vendorProject": "Microsoft", "product": "Windows", "vulnerabilityName": "Microsoft Windows Protection Mechanism Failure Vulnerability", "dateAdded": "2026-04-28", "dueDate": "2026-05-12", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-693"]}, {"cveID": "CVE-2025-29635", "vendorProject": "D-Link", "product": "DIR-823X", "vulnerabilityName": "D-Link DIR-823X Command Injection Vulnerability", "dateAdded": "2026-04-24", "dueDate": "2026-05-08", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-77"]}, {"cveID": "CVE-2024-7399", "vendorProject": "Samsung", "product": "MagicINFO 9 Server", "vulnerabilityName": "Samsung MagicINFO 9 Server Path Traversal Vulnerability", "dateAdded": "2026-04-24", "dueDate": "2026-05-08", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-22", "CWE-434"]}, {"cveID": "CVE-2024-57728", "vendorProject": "SimpleHelp ", "product": "SimpleHelp", "vulnerabilityName": "SimpleHelp Path Traversal Vulnerability", "dateAdded": "2026-04-24", "dueDate": "2026-05-08", "knownRansomwareCampaignUse": "Known", "cwes": ["CWE-22"]}, {"cveID": "CVE-2024-57726", "vendorProject": "SimpleHelp ", "product": "SimpleHelp", "vulnerabilityName": "SimpleHelp Missing Authorization Vulnerability", "dateAdded": "2026-04-24", "dueDate": "2026-05-08", "knownRansomwareCampaignUse": "Known", "cwes": ["CWE-862"]}, {"cveID": "CVE-2026-39987", "vendorProject": "Marimo", "product": "Marimo", "vulnerabilityName": "Marimo Remote Code Execution Vulnerability", "dateAdded": "2026-04-23", "dueDate": "2026-05-07", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-306"]}, {"cveID": "CVE-2026-33825", "vendorProject": "Microsoft", "product": "Defender", "vulnerabilityName": "Microsoft Defender Insufficient Granularity of Access Control Vulnerability", "dateAdded": "2026-04-22", "dueDate": "2026-05-06", "knownRansomwareCampaignUse": "Known", "cwes": ["CWE-1220"]}, {"cveID": "CVE-2026-20122", "vendorProject": "Cisco", "product": "Catalyst SD-WAN Manger", "vulnerabilityName": "Cisco Catalyst SD-WAN Manager Incorrect Use of Privileged APIs Vulnerability", "dateAdded": "2026-04-20", "dueDate": "2026-04-23", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-648"]}, {"cveID": "CVE-2026-20133", "vendorProject": "Cisco", "product": "Catalyst SD-WAN Manager", "vulnerabilityName": "Cisco Catalyst SD-WAN Manager Exposure of Sensitive Information to an Unauthorized Actor Vulnerability", "dateAdded": "2026-04-20", "dueDate": "2026-04-23", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-200"]}, {"cveID": "CVE-2025-2749", "vendorProject": "Kentico", "product": "Kentico Xperience", "vulnerabilityName": "Kentico Xperience Path Traversal Vulnerability", "dateAdded": "2026-04-20", "dueDate": "2026-05-04", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-22", "CWE-434"]}, {"cveID": "CVE-2023-27351", "vendorProject": "PaperCut", "product": "NG/MF", "vulnerabilityName": "PaperCut NG/MF Improper Authentication Vulnerability", "dateAdded": "2026-04-20", "dueDate": "2026-05-04", "knownRansomwareCampaignUse": "Known", "cwes": ["CWE-287"]}, {"cveID": "CVE-2025-48700", "vendorProject": "Synacor", "product": "Zimbra Collaboration Suite (ZCS)", "vulnerabilityName": "Synacor Zimbra Collaboration Suite (ZCS) Cross-site Scripting Vulnerability", "dateAdded": "2026-04-20", "dueDate": "2026-04-23", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-79"]}, {"cveID": "CVE-2026-20128", "vendorProject": "Cisco", "product": "Catalyst SD-WAN Manager", "vulnerabilityName": "Cisco Catalyst SD-WAN Manager Storing Passwords in a Recoverable Format Vulnerability", "dateAdded": "2026-04-20", "dueDate": "2026-04-23", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-257"]}, {"cveID": "CVE-2025-32975", "vendorProject": "Quest", "product": "KACE Systems Management Appliance (SMA)", "vulnerabilityName": "Quest KACE Systems Management Appliance (SMA) Improper Authentication Vulnerability", "dateAdded": "2026-04-20", "dueDate": "2026-05-04", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-287"]}, {"cveID": "CVE-2024-27199", "vendorProject": "JetBrains", "product": "TeamCity", "vulnerabilityName": "JetBrains TeamCity Relative Path Traversal Vulnerability", "dateAdded": "2026-04-20", "dueDate": "2026-05-04", "knownRansomwareCampaignUse": "Known", "cwes": ["CWE-23"]}, {"cveID": "CVE-2026-34197", "vendorProject": "Apache", "product": "ActiveMQ", "vulnerabilityName": "Apache ActiveMQ Improper Input Validation Vulnerability", "dateAdded": "2026-04-16", "dueDate": "2026-04-30", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-20", "CWE-94"]}, {"cveID": "CVE-2009-0238", "vendorProject": "Microsoft", "product": "Office", "vulnerabilityName": "Microsoft Office Remote Code Execution", "dateAdded": "2026-04-14", "dueDate": "2026-04-28", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-94"]}, {"cveID": "CVE-2026-32201", "vendorProject": "Microsoft", "product": "SharePoint Server", "vulnerabilityName": "Microsoft SharePoint Server Improper Input Validation Vulnerability", "dateAdded": "2026-04-14", "dueDate": "2026-04-28", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-20"]}, {"cveID": "CVE-2012-1854", "vendorProject": "Microsoft", "product": "Visual Basic for Applications (VBA)", "vulnerabilityName": "Microsoft Visual Basic for Applications Insecure Library Loading Vulnerability", "dateAdded": "2026-04-13", "dueDate": "2026-04-27", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-426"]}, {"cveID": "CVE-2025-60710", "vendorProject": "Microsoft", "product": "Windows", "vulnerabilityName": "Microsoft Windows Link Following Vulnerability", "dateAdded": "2026-04-13", "dueDate": "2026-04-27", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-59"]}, {"cveID": "CVE-2023-21529", "vendorProject": "Microsoft", "product": "Exchange Server", "vulnerabilityName": "Microsoft Exchange Server Deserialization of Untrusted Data Vulnerability", "dateAdded": "2026-04-13", "dueDate": "2026-04-27", "knownRansomwareCampaignUse": "Known", "cwes": ["CWE-502"]}, {"cveID": "CVE-2023-36424", "vendorProject": "Microsoft", "product": "Windows", "vulnerabilityName": "Microsoft Windows Out-of-Bounds Read Vulnerability", "dateAdded": "2026-04-13", "dueDate": "2026-04-27", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-125"]}]}''')

USE_LIVE_FEED = False
OFFICIAL_FEED = "https://www.cisa.gov/sites/default/files/feeds/known_exploited_vulnerabilities.json"

if USE_LIVE_FEED:
    try:
        with urllib.request.urlopen(OFFICIAL_FEED, timeout=30) as response:
            source_package = json.load(response)
    except (OSError, ValueError):
        raise RuntimeError("Feed download failed. Set USE_LIVE_FEED = False to use the teaching snapshot.") from None
    source_mode = "current official feed"
else:
    source_package = OFFLINE_SNAPSHOT
    source_mode = "embedded versioned teaching snapshot"

if not isinstance(source_package, dict):
    raise ValueError("Expected a catalog object. No database writes have run.")

print("Source mode:", source_mode)
print("Catalog version:", source_package.get("catalogVersion") or source_package.get("sourceCatalogVersion"))
print("Source release time:", source_package.get("dateReleased") or source_package.get("sourceDateReleased"))
print("Retrieval time:", source_package.get("retrievedAt", "live request in this runtime"))
print("Source URL:", source_package.get("sourceUrl", OFFICIAL_FEED))

### Normalize Only the Fields Used by This Lesson

The live feed contains more fields than the teaching snapshot. We intentionally
select the same compact fields in both paths. This is a modeling decision, not a
claim that omitted fields are unimportant.

In [ ]:
raw_records = source_package.get("vulnerabilities")
if not isinstance(raw_records, list) or not raw_records:
    raise ValueError("Expected a nonempty vulnerabilities list. Stop before loading.")
if not all(isinstance(raw, dict) for raw in raw_records[:75]):
    raise ValueError("Each selected vulnerability must be a JSON object.")

# Keep the lesson small even when the current feed is selected.
records = []
for raw in raw_records[:75]:
    records.append({
        "cveID": raw.get("cveID"),
        "vendorProject": raw.get("vendorProject"),
        "product": raw.get("product"),
        "vulnerabilityName": raw.get("vulnerabilityName"),
        "dateAdded": raw.get("dateAdded"),
        "dueDate": raw.get("dueDate"),
        "knownRansomwareCampaignUse": raw.get("knownRansomwareCampaignUse"),
        "cwes": raw.get("cwes"),
    })

print("Selected records:", len(records))
print("Selected fields:", list(records[0]))
print("Example record:\n", json.dumps(records[0], indent=2))

### Checks Before Any Database Write

A successful JSON parse does not establish that the records are suitable for a
database. These checks stop on empty required text, a malformed identifier,
duplicate IDs, invalid dates, or a non-list `cwes` value. We preserve the original
text, including trailing spaces. Silently changing a vendor name would change
what a later grouping means.

The dates describe calendar days. We keep canonical `YYYY-MM-DD` strings in JSON
and SQLite, and use PostgreSQL's `date` type in that path. An empty CWE list means
no entries in this field, not that a vulnerability has no weakness.

In [ ]:
fields = list(records[0])
required_text = fields[:-1]  # All selected fields except the CWE list.
quality_errors = []
whitespace_labels = []
for record in records:
    for field in required_text:
        value = record[field]
        if not isinstance(value, str) or not value.strip():
            quality_errors.append((field, "expected nonempty text"))
    if not re.fullmatch(r"CVE-[0-9]{4}-[0-9]{4,}", str(record["cveID"])):
        quality_errors.append(("cveID", "expected a CVE identifier"))
    for field in ("dateAdded", "dueDate"):
        try:
            parsed_date = date.fromisoformat(record[field])
            if parsed_date.isoformat() != record[field]:
                quality_errors.append((field, "use YYYY-MM-DD"))
        except (TypeError, ValueError):
            quality_errors.append((field, "invalid calendar date"))
    if not isinstance(record["cwes"], list) or not all(
        isinstance(value, str) and value.strip() for value in record["cwes"]
    ):
        quality_errors.append(("cwes", "expected a list of nonempty strings"))
    vendor = record["vendorProject"]
    if isinstance(vendor, str) and vendor != vendor.strip():
        whitespace_labels.append(repr(vendor))

if quality_errors:
    raise ValueError(f"Fix the source before loading: {quality_errors[:5]}")
cve_counts = Counter(record["cveID"] for record in records)
duplicate_ids = sorted(key for key, count in cve_counts.items() if count > 1)
if duplicate_ids:
    raise ValueError("Duplicate CVE identifiers. Stop and decide which record is authoritative.")
print("Passed required-text, ID, date, list, and uniqueness checks:", len(records))
print("Vendor labels with preserved leading/trailing spaces:", sorted(set(whitespace_labels)))

### Ask a Question Before Choosing a Database Shape

Our first question is: **Which vendors occur most often in this selected
snapshot?** This describes the sample, not all vulnerabilities and not a vendor's
security quality. The limited sample and source order matter.

In [ ]:
vendor_counts = Counter(record["vendorProject"] for record in records)
print("Five most frequent vendors in this selected sample:")
for vendor, count in sorted(vendor_counts.items(), key=lambda item: (-item[1], item[0]))[:5]:
    print(f"  {vendor!r}: {count}")

## 2. Measure Distribution Before Choosing Infrastructure

We compare four candidates:

- `vendorProject` can target vendor questions but may be skewed;
- `dateAdded` supports time questions but may be monotonic;
- `cveID` is highly distinct but does not target vendor questions; and
- `(vendorProject, product)` can divide some vendor groups further.

High cardinality alone is not enough. A useful decision also considers frequency,
write order, and actual query shapes.

In [ ]:
candidate_values = {
    "vendorProject": [record["vendorProject"] for record in records],
    "dateAdded": [record["dateAdded"] for record in records],
    "cveID": [record["cveID"] for record in records],
    "vendorProject + product": [
        (record["vendorProject"], record["product"]) for record in records
    ],
}

print(f"{'candidate':28} {'distinct':>8} {'ratio':>8} {'largest value share':>20}")
for name, values in candidate_values.items():
    frequencies = Counter(values)
    distinct = len(frequencies)
    largest_share = max(frequencies.values()) / len(values)
    print(f"{name:28} {distinct:8d} {distinct / len(values):8.2f} {largest_share:20.2%}")

dates_in_source_order = [record["dateAdded"] for record in records]
descending_date_order = all(
    left >= right for left, right in zip(dates_in_source_order, dates_in_source_order[1:])
)
print("\nDate-added values are monotonic descending in source order:", descending_date_order)
print("This describes input order, not production write order.")

### Range and Hashed Placement

This is not a sharded MongoDB deployment. It is a deterministic thought
experiment that makes two tradeoffs visible.

For the range simulation, the oldest 80 percent establishes three date boundaries
and the newest 20 percent acts like later writes. A monotonic time key tends to
place those later writes in the current high range. For the hash simulation,
SHA-256 maps CVE IDs into four teaching buckets. MongoDB uses its own hashing and
balancing behavior; these buckets only illustrate distribution.

In [ ]:
chronological = sorted(records, key=lambda record: record["dateAdded"])
split_at = max(1, int(len(chronological) * 0.80))
historical = chronological[:split_at]
later_writes = chronological[split_at:]

historical_dates = sorted(record["dateAdded"] for record in historical)
range_boundaries = [
    historical_dates[int(len(historical_dates) * fraction)]
    for fraction in (0.25, 0.50, 0.75)
]
range_buckets = Counter(
    bisect_right(range_boundaries, record["dateAdded"])
    for record in later_writes
)
hash_buckets = Counter(
    int(sha256(record["cveID"].encode("utf-8")).hexdigest(), 16) % 4
    for record in later_writes
)

print("Historical date boundaries:", range_boundaries)
print("Later-write range buckets:", dict(sorted(range_buckets.items())))
print("Later-write teaching hash buckets:", dict(sorted(hash_buckets.items())))
print("Later records tested:", len(later_writes))
print("These fixed teaching buckets do not model MongoDB range migrations or latency.")
assert sum(range_buckets.values()) == len(later_writes)
assert sum(hash_buckets.values()) == len(later_writes)

### Record a Capacity Recommendation

Use these questions to interpret the output before loading. They are discussion
prompts, not a separate written submission:

1. **Current scale:** This sample contains ___ records and is/is not large enough
   to justify sharding because ___.
2. **Candidate results:** ___ has ___ distinct values; its largest value holds
   ___ percent of the sample.
3. **Query targeting:** The main question filters/groups by ___, so ___ would or
   would not help route that question.
4. **Tradeoff:** Range distribution preserves ___ but risks ___; hashed
   distribution improves ___ but weakens ___.
5. **Decision:** Do not shard yet, or select ___ only under the stated future
   workload, because ___.

## 3. Import, Repeat, and Check the Result

The default is SQLite. It gives every student a complete database path without an
account. Set `TARGET` to `"sqlite"`, `"atlas"`, or `"postgres"`. Only one is
required. To try a second target, run cleanup first and then start this section
again. Cloud paths prompt for a URI without displaying it.

An **idempotent** load can be rerun without adding duplicate logical records. We
use `cveID` as the stable key and an upsert or conflict update on each target.
For an unchanged source, a second import should leave the same records and values.
This lesson updates records present in the source. It does not delete records
that disappear from a later feed, or keep a history of previous versions.

In [ ]:
TARGET = "sqlite"  # Alternatives: "atlas" or "postgres".
if TARGET not in {"sqlite", "atlas", "postgres"}:
    raise ValueError("Choose sqlite, atlas, or postgres.")
if globals().get("practice_active", False):
    raise RuntimeError("This run is still open. Repeat its IMPORT cell, or run cleanup first.")

# Keep this run's names when repeating an import.
run_id = uuid4().hex[:8]
atlas_database_name = "cst4714_public_data_" + run_id
postgres_schema = "cst4714_data_" + run_id
atlas_client = None
postgres_connection = None
con = None
rows = [
    (
        record["cveID"], record["vendorProject"], record["product"],
        record["vulnerabilityName"], record["dateAdded"], record["dueDate"],
        record["knownRansomwareCampaignUse"], json.dumps(record["cwes"]),
    )
    for record in records
]

practice_active = True
if TARGET == "sqlite":
    con = sqlite3.connect(":memory:")
    con.execute('''
        CREATE TABLE kev_sample (
            cve_id TEXT PRIMARY KEY,
            vendor_project TEXT NOT NULL,
            product TEXT NOT NULL,
            vulnerability_name TEXT NOT NULL,
            date_added TEXT NOT NULL,
            due_date TEXT NOT NULL,
            ransomware_use TEXT,
            cwes_json TEXT NOT NULL
        )
    ''')

### Import Into the Existing SQLite Table

The previous cell creates the practice environment. The next cell loads records
into that **same** environment. Run this import cell twice without recreating the
connection. The primary key identifies a vulnerability; `ON CONFLICT` updates it
instead of appending a duplicate.

In [ ]:
if TARGET == "sqlite":
    before_count = con.execute("SELECT count(*) FROM kev_sample").fetchone()[0]
    with con:
        con.executemany('''
        INSERT INTO kev_sample VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        ON CONFLICT (cve_id) DO UPDATE SET
            vendor_project = excluded.vendor_project,
            product = excluded.product,
            vulnerability_name = excluded.vulnerability_name,
            date_added = excluded.date_added,
            due_date = excluded.due_date,
            ransomware_use = excluded.ransomware_use,
            cwes_json = excluded.cwes_json
        ''', rows)
    after_count = con.execute("SELECT count(*) FROM kev_sample").fetchone()[0]
    print("SQLite records before / after:", before_count, after_count)

### Optional Atlas Path

Run this path only when `TARGET = "atlas"`. Open your own Atlas Free cluster and
create a **database user**, which is separate from your Atlas website login.
The next cell prints this Python runtime's public IPv4 address. In Atlas, open
**Database & Network Access > IP Access List** and add that address with `/32`.
Set an expiration if available. The browser's **Add Current IP** button may add
your laptop's address instead of Colab's address.

After adding the rule, return here and press Enter. Paste the Python connection
string from **Connect > Drivers** into the hidden prompt, with the database
user's credentials filled in. URI-reserved password characters must be percent
encoded. The cell keeps TLS certificate and hostname checks enabled and pings
before creating the practice collection. A successful ping proves connectivity,
not that records were loaded. If the runtime restarts, check its IP again.

The driver installation needs internet access. Connection failures display a
short explanation instead of a credential-bearing driver message. The temporary
URI variable is cleared after the attempt, but the connected client still holds
authentication state until cleanup closes it.

In [ ]:
if TARGET == "atlas" and atlas_client is None:
    %pip -q install pymongo
    from pymongo import MongoClient
    from pymongo.server_api import ServerApi

    try:
        with urllib.request.urlopen("https://api4.ipify.org", timeout=10) as response:
            runtime_ip = str(IPv4Address(response.read().decode().strip()))
    except Exception:
        raise RuntimeError("Could not obtain the runtime IPv4. Retry before connecting.") from None
    print("Add this runtime address to Atlas:", runtime_ip + "/32")
    input("After the Atlas IP rule is active, press Enter here: ")
    atlas_uri = getpass("Atlas mongodb+srv URI (hidden): ")
    try:
        if not atlas_uri.startswith("mongodb+srv://"):
            raise ValueError("Use the Atlas Drivers SRV string.")
        atlas_client = MongoClient(
            atlas_uri, tls=True, tlsInsecure=False,
            server_api=ServerApi("1", strict=True, deprecation_errors=True),
            serverSelectionTimeoutMS=10000, timeoutMS=10000,
        )
        atlas_client.admin.command("ping")
    except Exception:
        if atlas_client is not None:
            atlas_client.close()
        atlas_client = None
        raise RuntimeError(
            "Atlas connection failed. Check the database user, encoded password, "
            "cluster status, runtime /32 rule, DNS, and TLS. Keep TLS checks enabled."
        ) from None
    finally:
        atlas_uri = None
    atlas_collection = atlas_client[atlas_database_name]["kev_sample"]
    atlas_collection.create_index("cveID", unique=True)
    print("Atlas ping succeeded. Ready to import into:", atlas_database_name)

### Import Into the Existing Atlas Collection

Run this cell twice, without rerunning the target-setup cell. Each replacement
selects one stable CVE identifier. `upsert=True` inserts when that ID is absent.
The unique index prevents two documents with the same business identifier.
Each replacement is atomic for its document. This loop is not an all-or-nothing
transaction across all 75 documents. After an interrupted import, rerun it with
the same checked source and then verify the result.

In [ ]:
if TARGET == "atlas":
    before_count = atlas_collection.count_documents({})
    for record in records:
        atlas_collection.replace_one({"cveID": record["cveID"]}, record, upsert=True)
    after_count = atlas_collection.count_documents({})
    print("Atlas records before / after:", before_count, after_count)

### Optional Supabase/PostgreSQL Path

Run this path only when `TARGET = "postgres"`. In Supabase's **Connect** panel,
select the **Session pooler** connection string. Its shared pooler supports IPv4,
including notebook networks that cannot reach a direct IPv6 endpoint. Use the
database password and the exact pooler username from the panel, not the Supabase
website password or API key. Enter the URL only in the hidden prompt.

The URI must request `sslmode=require`, `verify-ca`, or `verify-full`. `require`
provides encrypted transport but does not guarantee server identity verification.
Use `verify-full` with the provider's CA certificate for certificate and hostname
verification. This notebook does not disable TLS to solve a connection problem.

The next cell creates a uniquely named practice schema. `sql.Identifier` quotes
that schema name correctly; `%s` placeholders are for **values**, not table names.
A quoted identifier and a value parameter solve different problems.
Autocommit keeps the connection out of an idle transaction between cells. Each
`transaction()` block still commits its complete operation on success or rolls it
back on an error. `SET LOCAL search_path` applies only inside that transaction.

In [ ]:
if TARGET == "postgres" and postgres_connection is None:
    %pip -q install "psycopg[binary]"
    import psycopg
    from psycopg import sql
    from psycopg.conninfo import conninfo_to_dict

    postgres_url = getpass("PostgreSQL connection URL (hidden): ")
    try:
        sslmode = conninfo_to_dict(postgres_url).get("sslmode", "prefer")
        if sslmode not in {"require", "verify-ca", "verify-full"}:
            raise ValueError("Encrypted transport is required.")
        postgres_connection = psycopg.connect(
            postgres_url, connect_timeout=10, autocommit=True, prepare_threshold=None
        )
        with postgres_connection.transaction():
            with postgres_connection.cursor() as cursor:
                cursor.execute("SET LOCAL statement_timeout = '10s'")
                cursor.execute(sql.SQL("CREATE SCHEMA {}").format(sql.Identifier(postgres_schema)))
                cursor.execute(sql.SQL("SET LOCAL search_path TO {}").format(
                    sql.Identifier(postgres_schema)))
                cursor.execute('''
                CREATE TABLE kev_sample (
                    cve_id text PRIMARY KEY,
                    vendor_project text NOT NULL,
                    product text NOT NULL,
                    vulnerability_name text NOT NULL,
                    date_added date NOT NULL,
                    due_date date NOT NULL,
                    ransomware_use text,
                    cwes jsonb NOT NULL
                )
                ''')
    except Exception:
        if postgres_connection is not None:
            postgres_connection.close()
        postgres_connection = None
        raise RuntimeError(
            "PostgreSQL connection/setup failed. Check the session-pooler URL, "
            "database password, project status, permissions, and sslmode=require "
            "or verify-full with its CA certificate."
        ) from None
    finally:
        postgres_url = None
    print("PostgreSQL is connected. Practice schema:", postgres_schema)

### Import Into the Existing PostgreSQL Table

Run this import cell twice. The transaction covers the complete batch. If one row
fails, this transaction rolls back instead of leaving a half-imported batch.
`excluded` names the proposed row that collided with the primary key. `jsonb`
preserves the CWE list instead of squeezing multiple values into one delimited
text label. SQL values travel separately from the SQL command.

In [ ]:
if TARGET == "postgres":
    with postgres_connection.transaction():
        with postgres_connection.cursor() as cursor:
            cursor.execute("SET LOCAL statement_timeout = '10s'")
            cursor.execute(sql.SQL("SET LOCAL search_path TO {}").format(
                sql.Identifier(postgres_schema)))
            cursor.execute("SELECT count(*) FROM kev_sample")
            before_count = cursor.fetchone()[0]
            cursor.executemany('''
                INSERT INTO kev_sample VALUES
                    (%s, %s, %s, %s, %s, %s, %s, %s::jsonb)
                ON CONFLICT (cve_id) DO UPDATE SET
                    vendor_project = excluded.vendor_project,
                    product = excluded.product,
                    vulnerability_name = excluded.vulnerability_name,
                    date_added = excluded.date_added,
                    due_date = excluded.due_date,
                    ransomware_use = excluded.ransomware_use,
                    cwes = excluded.cwes
            ''', rows)
            cursor.execute("SELECT count(*) FROM kev_sample")
            after_count = cursor.fetchone()[0]
    print("PostgreSQL records before / after:", before_count, after_count)

### Verify Values as Well as Counts

We verify four things:

1. expected and observed counts;
2. every selected field for one known identifier, including dates and the CWE list;
3. a grouped question tied to the reason for loading; and
4. rerun behavior through the primary key and upsert.

These checks do not establish completeness of the full live catalog, correct
authorization, backup readiness, performance under load, or production fitness.

In [ ]:
expected_known = records[0]
known_id = expected_known["cveID"]

if TARGET == "sqlite":
    observed_count = con.execute("SELECT count(*) FROM kev_sample").fetchone()[0]
    known_row = con.execute(
        "SELECT * FROM kev_sample WHERE cve_id = ?",
        [known_id],
    ).fetchone()
    assert known_row is not None, "The selected source ID is missing."
    observed_known = dict(zip(fields, known_row))
    observed_known["cwes"] = json.loads(observed_known["cwes"])

if TARGET == "atlas":
    observed_count = atlas_collection.count_documents({})
    observed_known = atlas_collection.find_one(
        {"cveID": known_id}, {"_id": 0}
    )

if TARGET == "postgres":
    with postgres_connection.transaction():
        with postgres_connection.cursor() as cursor:
            cursor.execute("SET LOCAL statement_timeout = '10s'")
            cursor.execute(sql.SQL("SET LOCAL search_path TO {}").format(
                sql.Identifier(postgres_schema)))
            cursor.execute("SELECT count(*) FROM kev_sample")
            observed_count = cursor.fetchone()[0]
            cursor.execute(
                "SELECT * FROM kev_sample WHERE cve_id = %s",
                (known_id,),
            )
            known_row = cursor.fetchone()
    assert known_row is not None, "The selected source ID is missing."
    observed_known = dict(zip(fields, known_row))
    for field in ("dateAdded", "dueDate"):
        observed_known[field] = observed_known[field].isoformat()

print("Expected / observed records:", len(records), observed_count)
print("Known source record:", expected_known)
print("Stored record:", observed_known)
assert observed_count == len(records), "Record count differs from the checked source."
assert observed_known == expected_known, "A stored value differs, even if the count is correct."
print("The selected record matches all eight source fields.")

### Your Query: Vendor or Product Groups

Run the vendor question once. Then change `GROUP_FIELD` to `"product"` and rerun
this cell. The question changes from vendor counts to product counts. State what
one result row now means. These are counts of catalog entries in this selected
sample, not numbers of affected computers or a comparison of security quality.

The two-entry mapping restricts the selectable SQL column. SQLite's f-string
below receives only one of those approved column names. Do not put unrestricted
input there. PostgreSQL uses `sql.Identifier`, while the Atlas pipeline builds a
field expression such as `$product`. The known-ID query above instead uses a
value parameter. A placeholder for a value cannot choose a column name.

The last lines compute the same grouping directly from the checked source. A
tie breaks by the group label so that both results have a defined order. This
independent comparison is stronger than deciding that a plausible-looking table
must be right.

In [ ]:
GROUP_FIELD = "vendorProject"  # Student change: "product".
allowed_columns = {"vendorProject": "vendor_project", "product": "product"}
if GROUP_FIELD not in allowed_columns:
    raise ValueError("Choose vendorProject or product.")
group_column = allowed_columns[GROUP_FIELD]

if TARGET == "sqlite":
    grouped = con.execute(f'''
        SELECT {group_column}, count(*) AS vulnerability_count
        FROM kev_sample GROUP BY {group_column}
        ORDER BY vulnerability_count DESC, {group_column} COLLATE BINARY
        LIMIT 5
    ''').fetchall()

if TARGET == "atlas":
    result = list(atlas_collection.aggregate([
        {"$group": {"_id": "$" + GROUP_FIELD, "n": {"$sum": 1}}},
        {"$sort": {"n": -1, "_id": 1}},
        {"$limit": 5},
    ]))
    grouped = [(row["_id"], row["n"]) for row in result]

if TARGET == "postgres":
    with postgres_connection.transaction():
        with postgres_connection.cursor() as cursor:
            cursor.execute("SET LOCAL statement_timeout = '10s'")
            cursor.execute(sql.SQL("SET LOCAL search_path TO {}").format(
                sql.Identifier(postgres_schema)))
            cursor.execute(sql.SQL('''
                SELECT {field}, count(*) AS vulnerability_count
                FROM kev_sample GROUP BY {field}
                ORDER BY vulnerability_count DESC, {field} COLLATE "C"
                LIMIT 5
            ''').format(field=sql.Identifier(group_column)))
            grouped = cursor.fetchall()

source_groups = Counter(record[GROUP_FIELD] for record in records)
expected_groups = sorted(source_groups.items(), key=lambda item: (-item[1], item[0]))[:5]
print("Grouped field:", GROUP_FIELD)
for label, count in grouped:
    print(f"{label!r}: {count}")
assert grouped == expected_groups, "Database and source groupings disagree."
print("The five displayed groups match the checked source.")

## Submission Record

Write one short handoff for someone who will maintain this import. Explain the
question you asked, why your key choice fits or does not fit it, and what happened
when you reran the import into the same target. Use two capacity measurements and
one verification result already shown above. Interpret one row from your changed
product grouping. Acknowledge the historical subset and one limitation.

The imagined maintainer is the audience for your writing, not a required partner.
Work individually. Submit this notebook in Brightspace with the changed query,
outputs, and handoff. There is no separate Day 1 submission or second report.

Before submitting, remove any accidentally saved credential from source or output.

## Final-Project Transfer

This checkpoint does not add or redefine final-project deliverables. Use the
final-project assignment supplied in Brightspace or the course package.

Consider these prompts during project work. No additional written response is
required:

1. My project could use a public or synthetic dataset about ___.
2. The source of truth should be ___ because ___.
3. My first scale trigger would be measured as ___.
4. Before scaling, I would verify ___ and improve ___ because ___.

The cleanup cell drops only this run's `kev_sample` collection or table, then
closes its connection. An unrelated collection or table survives. PostgreSQL
removes the generated schema only if it is empty. For Atlas, also remove the
temporary runtime `/32` IP rule in the dashboard. Closing a Python client does
not change that network rule or pause a hosted database.

In [ ]:
if atlas_client is not None:
    try:
        atlas_client[atlas_database_name].drop_collection("kev_sample")
        print("Removed this run's Atlas collection. Other collections were kept.")
    finally:
        atlas_client.close()
        atlas_client = None
    print("Remove the temporary runtime /32 rule in Atlas after class.")
if postgres_connection is not None:
    try:
        with postgres_connection.cursor() as cursor:
            cursor.execute(sql.SQL("DROP TABLE IF EXISTS {}.kev_sample").format(
                sql.Identifier(postgres_schema)))
            try:
                cursor.execute(sql.SQL("DROP SCHEMA IF EXISTS {}").format(
                    sql.Identifier(postgres_schema)))
            except psycopg.errors.DependentObjectsStillExist:
                print("Kept the practice schema because it contains other objects.")
        print("Removed this run's PostgreSQL table. No CASCADE was used.")
    finally:
        postgres_connection.close()
        postgres_connection = None
if con is not None:
    con.close()
    con = None
    print("Closed the disposable SQLite database.")
practice_active = False